In [1]:
import os
import pandas as pd
from html import escape as html_escape

In [11]:
# Load your Excel file into a DataFrame
publications = pd.read_excel("pubs.xlsx")

# Initialize a list to hold the content of the Markdown page
md_content = [
    "---",
    "title: \"Publications\"",
    "layout: single",  # or another layout depending on your static site generator
    "collection: publications",
    "permalink: /publications/",
    "---"
]

for _, item in publications.iterrows():
    publication_entry = [
        f"- {html_escape(item.citation)}"
    ]

    # Include the abstract if available
    if pd.notna(item.abstract) and len(str(item.abstract)) > 5:
        # Limit to 25 words in the default view
        abstract_words = str(item.abstract).split()
        short_abstract = " ".join(abstract_words[:25])
        remaining_abstract = " ".join(abstract_words[25:])

        if remaining_abstract:
                # Create expandable/collapsible HTML for abstract
            publication_entry.append(
                f"""  - **Abstract**: <span style="font-size:small;">{html_escape(short_abstract)}... 
                <a href="javascript:void(0)" onclick="this.nextElementSibling.style.display='block'; this.style.display='none'; this.previousElementSibling.style.display='none';">(read more)</a>
                <span style="display:none;">{html_escape(remaining_abstract)} <a href="javascript:void(0)" onclick="this.parentElement.style.display='none'; this.parentElement.previousElementSibling.style.display='inline'; this.parentElement.previousElementSibling.previousElementSibling.style.display='inline';">(collapse)</a></span></span>"""
            )
        else:
                # If the abstract is already within the 25-word limit
            publication_entry.append(
                f"""  - **Abstract**: <span style="font-size:small;">{html_escape(short_abstract)}</span>"""
            )

    # Include a link to the full text if available
    if pd.notna(item.fulltext_url) and len(str(item.fulltext_url)) > 5:
        publication_entry.append(f"  - [Link to full text]({item.fulltext_url})")

    # Include a link to the slides if available
    if pd.notna(item.slides_url) and len(str(item.slides_url)) > 5:
        publication_entry.append(f"  - [Link to slides]({item.slides_url})")

    # Add a blank line for spacing between publications
    publication_entry.append("\n")

    # Append the entry to the Markdown content list
    md_content.extend(publication_entry)

# Join all the content into a single string
final_md_content = "\n".join(md_content)

# Save the content to a Markdown file
output_filename = "../_publications/all_publications.md"
os.makedirs(os.path.dirname(output_filename), exist_ok=True)

with open(output_filename, 'w') as f:
    f.write(final_md_content)